# Hyperparameter Tuning

## Goals

The overarching goal is to learn to use callbacks for some typical tasks. These include:
- Reporting about training progress.
- Stoping once training no longer reduces loss.
- Tuning hyperparameters.
- Implementing adaptive learning rate decay.
- Finding an optimal batch-size for training.
- Putting some of this into ```my_keras_utils.py``` so that they can be easily called and reused.

## What's Here?

I continue working with MNIST data, which I began working with in [my first Keras models](first_model.ipynb). 

My **concrete objective** is to tune a model that does well on Kaggle: 97th percentile? That's tough, but I think I can make it work.

In [1]:
import numpy as np
from datetime import datetime, time, timedelta

import pandas as pd
import tensorflow as tf
import kerastuner as kt
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt


import my_keras_utils as my_utils

In [2]:
tf.__version__
tf.config.experimental.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [27]:
## Load our data.
## Since the load process is a little slow, the try-except allows us to re-run all 
## cells without having to wait. 
try:
    ## Raises NameError and loads data if X_train is not defined.
    X_train.shape
except NameError: 
    ((X_train, y_train), (X_dev, y_dev), (X_test, y_test)) = my_utils.load_kaggle_mnist()

## Reshape the flattened rows into (-1,28,28,1)
## For augmentation
X_train = X_train.reshape((-1,28,28,1))    
X_dev = X_dev.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

In [3]:

def overlay_histories(histories, metric):
    fig, ax = plt.subplots()
    n = 0
    for h in histories:
        x = range(0,len(h.history[metric]))
        y = np.array(h.history[metric])
        label = 'history_' + str(n)
        ax.plot(x,y, label=label)
        n += 1
    ax.legend()

## Augmentation Model

In [35]:
augment_model = keras.models.Sequential(name= 'augment_laygit ')
augment_model.add(layers.experimental.preprocessing
                    .RandomRotation(factor = 1./60.,
                                    fill_mode = 'constant'))
augment_model.add(layers.experimental.preprocessing
                    .RandomTranslation(height_factor=1./28.,
                                    width_factor= 1./28.,
                                    fill_mode = 'constant'))
augment_model.add(layers.Flatten())

## Tuning

Time to work with Keras Tuner.



In [36]:
def model_builder(hp):
    ## ### I need to learn about the options here. 'Choice' means "here are your choices"
    ## ### 'Int' is a different option that searches an integer range by steps.

    inputs = keras.Input(shape=(28,28,1))
    x = augment_model(inputs)
    x = layers.experimental.preprocessing.Rescaling(1./255)(x)
    x = layers.Dropout(rate = hp.Float('drop_rate_0',
                                min_value = .10,
                                max_value = .45,
                                step = .02,
                                default = .25))(x)
    for i in range(hp.Int('num_layers', min_value = 2, max_value = 3, default = 2)):
        hp_units = hp.Int('units_l'+str(i+1), 
                            min_value = 80, 
                            max_value = 120, 
                            step = 20,
                            default = 100)
        x = layers.Dense(hp_units, activation = 'relu')(x)
        hp_dropout = hp.Float('drop_rate_'+str(i+1),
                                min_value = .10,
                                max_value = .45,
                                step = .01,
                                default = .15)
        x = layers.Dropout(rate=hp_dropout)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer = keras.optimizers
                                .Adam(hp.Float('learning_rate',
                                                min_value = .0003,
                                                max_value = .01,
                                                sampling = 'log',
                                                default = .001)),
                    loss = "sparse_categorical_crossentropy",
                    run_eagerly = False,
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )

    return model

	


NameError: name 'model' is not defined

In [37]:
## Tuner callbacks
clear_output = my_utils.ClearTrainingOutput()
timed_update = my_utils.TimedProgressUpdate(3)
train_loss_stopping = keras.callbacks.EarlyStopping(monitor='loss', 
                            patience = 10, 
                            restore_best_weights = False
                            )

adaptive_lr = keras.callbacks.ReduceLROnPlateau(
                    monitor='loss',
                    patience= 2,
                    cooldown= 2,
                    min_lr=.0003, 
                    factor=0.334,)

tuner_callbacks = [adaptive_lr, train_loss_stopping, clear_output]

In [39]:

hp = kt.HyperParameters()
#hp.Int('num_layers', 
#        min_value = 2,
#        max_value = 3, 
#        default = 2)
hp.Fixed('num_layers', 3)
hp.Float('learning_rate',
            min_value = .0003,
            max_value = .015,
            sampling = 'log',
            default = .001)
assert 'num_layers' in hp

model_builder(hp).summary()

test_tuner = kt.RandomSearch(model_builder,
                objective = 'val_loss',
                hyperparameters= hp,
                max_trials = 10,
                executions_per_trial = 1,
                tune_new_entries = False,
                directory = os.path.normpath('mnist'),
                project_name = 'test',
                overwrite = True)
                

##assert 'keepgoing' == False

test_tuner.search(X_train,
                y_train,
                validation_data = (X_dev, y_dev),
                epochs = 7,
                batch_size = 64,
                verbose = 1,
                callbacks = [train_loss_stopping])

Trial 10 Complete [00h 00m 38s]
val_loss: 0.10064239054918289

Best val_loss So Far: 0.10064239054918289
Total elapsed time: 00h 06m 08s
INFO:tensorflow:Oracle triggered exit


In [40]:
test_tuner.results_summary()

Results summary
Results in mnist\test
Showing 10 best trials
Objective(name='val_loss', direction='min')
Trial summary
Hyperparameters:
num_layers: 3
learning_rate: 0.0019241973072307247
drop_rate_0: 0.1
units_l1: 100
drop_rate_1: 0.4199999999999998
units_l2: 100
drop_rate_2: 0.14999999999999997
units_l3: 80
drop_rate_3: 0.14999999999999997
Score: 0.10064239054918289
Trial summary
Hyperparameters:
num_layers: 3
learning_rate: 0.0019183496863989474
drop_rate_0: 0.16000000000000003
units_l1: 100
drop_rate_1: 0.3199999999999999
units_l2: 100
drop_rate_2: 0.4199999999999998
units_l3: 100
drop_rate_3: 0.2599999999999999
Score: 0.10295113921165466
Trial summary
Hyperparameters:
num_layers: 3
learning_rate: 0.0003275545655470625
drop_rate_0: 0.22000000000000003
units_l1: 100
drop_rate_1: 0.33999999999999986
units_l2: 100
drop_rate_2: 0.11
units_l3: 100
drop_rate_3: 0.15999999999999998
Score: 0.1140589788556099
Trial summary
Hyperparameters:
num_layers: 3
learning_rate: 0.0016838186827172982
d

In [42]:
import math
epochs_ = 70
factor_ = 4
total_epochs = epochs_*math.log(epochs_,factor_)**2
print(total_epochs/60*2)

21.91473100938917


In [ ]:
test_tuner.results_summary()

In [9]:
hp = kt.HyperParameters()
hp.Fixed('learning_rate', .001)

hyperband = kt.Hyperband(model_builder,
                     objective = 'val_loss', 
                     max_epochs = 70,
                     hyperparameters= hp,
                     hyperband_iterations = 2,
                     factor = 2,
                     directory = os.path.normpath('mnist'),
                     project_name = 'hb',
                     )	

hyperband.search_space_summary()

In [10]:
hyperband.search(X_train, y_train, 
            epochs=70,
            batch_size=32, 
            validation_data=(X_dev, y_dev),
            callbacks = tuner_callbacks,
            verbose = 0
            )


INFO:tensorflow:Oracle triggered exit


In [12]:
hyperband.results_summary()

-drop_rate_1: 0.1

|-drop_rate_2: 0.34999999999999987

|-drop_rate_3: 0.13

|-drop_rate_4: 0.3999999999999998

|-learning_rate: 0.003

|-num_layers: 2

|-tuner/bracket: 1

|-tuner/epochs: 70

|-tuner/initial_epoch: 35

|-tuner/round: 1

|-tuner/trial_id: 90315f9690e8eeddbb36b78a2a64ed9e

|-units_l2: 120

|-units_l3: 120

|-units_l4: 100

## Ran the Hyperband, 

with the following params:
```
tuner = kt.Hyperband(model_builder,
                     objective = 'val_loss', 
                     max_epochs = 200,
                     hyperband_iterations = 10,
                     factor = 3,
                     directory = 'ignored/kt_trials',
                     project_name = 'dropout_mnist')	
###
tuner.search(X_train, y_train, 
            epochs=50,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = tuner_callbacks,
            )
```
Which involved way too many iterations. I didn't notice the hyperband_iterations param (10!).

### os.path.normpath

Per [Issue #198](https://github.com/keras-team/keras-tuner/issues/198) you may need to add os.path.normpath() to the directory keyword arg in Windows and the path to the logs needs to be short. (E.g., you won't be able to use my_trials/this_type_of_trial/this_trial--too long. Try mt/t/this).

## Baysian Tuning

Let's see how well the Baysian Optimizer does with the rand_search_builder.

In [ ]:
bayesian_tuner = kt.tuners.BayesianOptimization(
            rand_search_model_builder,
            objective='val_loss',
            max_trials = 25,
            directory = os.path.normpath('ignored/mnist'),
            project_name = 'bayes'
)

bayesian_tuner.search(X_train, y_train, 
            epochs=120,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        clear_output
                        ],
            verbose = 0
            )


In [ ]:
bayesian_tuner.results_summary()

## Let's put it all together. 

The Bayesian tuner got pretty bad results compared with the other two. I'm not really sure what happened, except that it didn't seem exploratory enough. I should probably figure out how to get better Bayesian results.

But for now, we have several high scoring models. Let's extract three and cross validate.  

In [ ]:
best = rand_tuner.get_best_models()[0]
config = best.get_config()
model_1 = best.from_config(config)
config

In [ ]:
temp = rand_tuner.get_best_models(num_models=2)
model_2 = temp[0].from_config(temp[0].get_config())
model_3 = temp[1].from_config(temp[1].get_config())


In [ ]:
def cross_validate(model, X, y, val_size = 2000, folds = 3):
    init_weights = model.get_weights()
    results = []
    for fold in range(folds):
        X_val = X[fold*val_size:(fold+1)*val_size,:]
        X_train = np.concatenate((X[0:fold*val_size,:],X[(fold+1)*val_size:,:]))
        y_val = y[fold*val_size:(fold+1)*val_size]
        y_train = np.concatenate((y[0:fold*val_size],y[(fold+1)*val_size:]))
        history = model.fit(X_train, y_train,
            epochs=200,
            batch_size=128, 
            validation_data=(X_val, y_val),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        clear_output
                        ],
            verbose = 0)
        model.set_weights(init_weights)
        results.append(history)
    return results

In [ ]:
results = []
for model in [model_1, model_2, model_3]:
    model.compile(optimizer = keras.optimizers.Adam(.003),
                    loss = "sparse_categorical_crossentropy",
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )
    histories = cross_validate(model, X_train, y_train)
    results.append(histories)

In [ ]:
for e in results[0]:
    print(e.history['val_loss'][-1])
for e in results[1]:
    print(e.history['val_loss'][-1])
for e in results[2]:
    print(e.history['val_loss'][-1])
    

In [ ]:
## Reset all the weights.
model_1.compile(optimizer = keras.optimizers.Adam(.003),
                    loss = "sparse_categorical_crossentropy",
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )
## Save so I can move to another computer
model_1.save('MNIST_model.hdf5')
model_2.save("mnist_2.hdf5")
model_3.save("mnist_3.hdf5")

In [ ]:
model_1.fit(X_train, y_train,
            epochs=200,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        clear_output
                        ],
            verbose = 2)
model_1.save('MNIST_model.hdf5')